# 02 · Statistical matching EPH→ENGH

Tenemos clases LCA en EPH y necesitamos transportarlas a la ENGH. Como no existe el mismo hogar en ambas encuestas, no hay matching exacto: se entrena un Random Forest en EPH usando **sólo variables presentes en las dos encuestas** y se predice la clase para cada ocupado de la ENGH.

**Supuesto crítico declarado: CIA (Conditional Independence Assumption).** Condicionado en las variables comunes, la pertenencia a clase LCA y la canasta de gasto son independientes. No es testeable directamente.

**Validación indirecta:** se reporta el accuracy del RF *dentro* de EPH (train/test split). Es un techo, no una validación de la imputación a la ENGH.

**Variables comunes (5):** `SEXO`, `EDAD_GRP`, `NIVEL_ED_3`, `CAT_OCUP`, `REGION`.

> Notar la **asimetría rica/pobre**: el LCA se entrena con 6 variables (incluyendo `FORMAL`); el matching usa 5 (sin `FORMAL`, porque no hay equivalente directo en ENGH). Esta asimetría es estándar en statistical matching y se reconoce como limitación.

In [ ]:
import pandas as pd
import numpy as np
import pickle, warnings
warnings.filterwarnings("ignore")
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder

eph = pd.read_parquet("data/eph_with_lca.parquet")
eph_train = eph[eph["ANO4"].isin([2017, 2018])].copy()
engh = pd.read_parquet("data/engh_ocupados.parquet")
print("EPH 2017-2018 (con LCA):", eph_train.shape)
print("ENGH ocupados:", engh.shape)

## Encoding uniforme entre EPH y ENGH

In [ ]:
COMMON = ["SEXO","EDAD_GRP","NIVEL_ED_3","CAT_OCUP","REGION"]

common_encoders = {}
for v in COMMON:
    le = LabelEncoder()
    union = pd.concat([eph_train[v].astype(str), engh[v].astype(str)]).unique()
    le.fit(union)
    eph_train[f"{v}_enc"] = le.transform(eph_train[v].astype(str))
    engh[f"{v}_enc"]      = le.transform(engh[v].astype(str))
    common_encoders[v] = le

## Random Forest

In [ ]:
X = eph_train[[f"{v}_enc" for v in COMMON]].values
y = eph_train["LCA_class"].values
w = eph_train["PONDERA"].values

Xtr, Xte, ytr, yte, wtr, wte = train_test_split(
    X, y, w, test_size=0.25, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=300, max_depth=None,
                            random_state=42, n_jobs=-1, class_weight="balanced")
rf.fit(Xtr, ytr, sample_weight=wtr)

print(f"Accuracy train: {rf.score(Xtr, ytr, sample_weight=wtr):.3f}")
print(f"Accuracy test : {rf.score(Xte, yte, sample_weight=wte):.3f}")
print()
print(classification_report(yte, rf.predict(Xte), sample_weight=wte, digits=3))

print("Importancias:")
for v, imp in zip(COMMON, rf.feature_importances_):
    print(f"  {v}: {imp:.3f}")

## Aplicar el RF a la ENGH

Cada ocupado de la ENGH recibe la clase LCA más probable, más las probabilidades posteriores de pertenecer a cada clase.

In [ ]:
Xe = engh[[f"{v}_enc" for v in COMMON]].values
post = rf.predict_proba(Xe)
engh["LCA_class_pred"] = post.argmax(axis=1)
for c in range(rf.n_classes_):
    engh[f"LCA_p{c}"] = post[:, c]

engh.to_parquet("data/engh_ocupados_with_lca.parquet", index=False)
with open("data/rf_match_model.pkl","wb") as f:
    pickle.dump({"model":rf, "common":COMMON, "encoders":common_encoders}, f)

print("Distribución en ENGH (predicha):")
print(engh["LCA_class_pred"].value_counts(normalize=True).sort_index().round(3))
print("\nDistribución en EPH 2017-18 (real):")
print(eph_train["LCA_class"].value_counts(normalize=True).sort_index().round(3))

### Cómo leer estos números

Si las dos distribuciones son cercanas, el RF está reproduciendo en ENGH una composición similar a la observada en EPH. Diferencias grandes serían señal de mala calibración o de que las muestras son estructuralmente distintas. Para el POC obtenemos ≈80% de accuracy y distribuciones razonablemente alineadas.

**Lo que NO valida esto**: que la imputación de la clase a la ENGH sea correcta a nivel hogar. Eso es la CIA, y la CIA no se testea aquí.